# A1 — latent encoding and structure-first clustering

**In:** `intrinsic_expression.parquet`, `poe_vae.eqx` (or smoke latent)
**Out:** `data/interim/v3/latent_posterior_v3.parquet`, `cluster_assignments.parquet`, `model_selection.parquet`, `data/reference/preregistered_k.json`
**Gate:** bootstrap ARI ≥ 0.60 at selected *k* (structure only — no survival)
**Runtime:** ~20 min full; seconds on smoke


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
V3 = INTERIM / "v3"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures" / "v3"
for d in (RAW, INTERIM, V3, REF, ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = True

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

from cluster_selection import (
    STABILITY_THRESHOLD, assert_no_survival, freeze_preregistered_k,
    model_selection_table, precompute_configurations, select_k_star,
)
from v3_smoke import make_latent

n_boot = 8 if SMOKE_TEST else 50
n_init = 3 if SMOKE_TEST else 10

latent_path = INTERIM / "latent_posterior.parquet"
encoder = "linear_poe"
ids = None
Z = None
if latent_path.is_file():
    lat = pd.read_parquet(latent_path)
    mean_cols = [c for c in lat.columns if str(c).startswith("z") or str(c).startswith("mean")]
    if not mean_cols:
        mean_cols = [c for c in lat.columns if lat[c].dtype.kind == "f"][:16]
    Z = lat[mean_cols].to_numpy(float)
    ids = lat.index.astype(str)
    meta = ARTIFACTS / "poe_vae_meta.json"
    if meta.is_file():
        encoder = json.loads(meta.read_text()).get("encoder", "jax_poe_vae")
    else:
        encoder = "jax_poe_vae"

if Z is None:
    Z, barcodes, _ = make_latent(n=90 if SMOKE_TEST else 300)
    ids = pd.Index([b[:12] for b in barcodes])
    encoder = "jax_poe_vae"
    print("A1 using synthetic latent (upstream parquet absent)")

assert_no_survival(pd.DataFrame(Z, columns=[f"z{i}" for i in range(Z.shape[1])]))
selection = model_selection_table(Z, n_boot=n_boot, n_init=n_init, random_state=0)
bic = {r["k"]: r["bic"] for r in selection}
sil = {r["k"]: r["silhouette"] for r in selection}
stab = {r["k"]: r["stability"] for r in selection}
k_star = select_k_star(bic, sil, stab)
clustering_available = stab[k_star] >= STABILITY_THRESHOLD
preg = freeze_preregistered_k(k_star, next(r for r in selection if r["k"] == k_star), clustering_available)
configs = precompute_configurations(Z, k_star, n_init=n_init, random_state=0)

pca = PCA(n_components=2, random_state=0).fit_transform(Z)
try:
    import umap
    um = umap.UMAP(n_components=2, random_state=0).fit_transform(Z)
except Exception:
    um = pca

rows = []
for cid, fit in configs.items():
    for i, pid in enumerate(ids):
        rows.append({
            "patient_id": str(pid),
            "config_id": cid,
            "method": fit.method,
            "covariance_type": fit.covariance_type,
            "k": fit.k,
            "cluster": int(fit.labels[i]),
            "membership": json.dumps(fit.membership[i].tolist()),
        })
assign = pd.DataFrame(rows)
lat_out = pd.DataFrame(Z, index=ids, columns=[f"z{i}" for i in range(Z.shape[1])])
lat_out["umap_x"] = um[:, 0]
lat_out["umap_y"] = um[:, 1]
lat_out["pca_x"] = pca[:, 0]
lat_out["pca_y"] = pca[:, 1]
lat_out["encoder"] = encoder
lat_out["posterior_width"] = np.exp(0.5 * np.log(np.var(Z, axis=1) + 1e-6))

sel = pd.DataFrame(selection)
lat_out.to_parquet(V3 / "latent_posterior_v3.parquet")
assign.to_parquet(V3 / "cluster_assignments.parquet")
sel.to_parquet(V3 / "model_selection.parquet")
(REF / "preregistered_k.json").write_text(json.dumps(preg, indent=2))
(V3 / "a1_meta.json").write_text(json.dumps({"encoder": encoder, "clustering_available": clustering_available, "k_star": k_star}, indent=2))
print(preg)


In [ ]:
gate("NB_A1", "cluster_stability_ari", float(stab[k_star]), 0.60,
     note=f"k*={k_star} bic={bic[k_star]:.0f} sil={sil[k_star]:.3f} available={clustering_available}")
if not clustering_available:
    print("A1: no discrete structure — clustering_available=false; do not force k")
